In [36]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
from datasets import load_dataset
import pandas as pd
from typing import List
from data_loader import fetch_categories_mmlu, load_mmlu_dataset

## Loading the MMLU dataset
normative_category = [
    "moral_disputes",
    "philosophy",
    "world_religions",
    "us_foreign_policy",
    "sociology",
    "professional_psychology",
    "professional_law",
    "moral_scenarios",
    "human_sexuality",
    "international_law",
]

control_category = ["college_mathematics",
    "college_physics",
    "formal_logic",
    "logical_fallacies",
    "college_computer_science",
]

dataset_3_subjects = normative_category + control_category

mmlu_full_df = fetch_categories_mmlu(dataset_3_subjects)

samples_per_subject = 50
samples_examples_per_subject = 5

sample_mmlu, sample_examples_mmlu = load_mmlu_dataset(
    mmlu_full_df,
    sample_per_subject=samples_per_subject,
    sample_examples_per_subject=samples_examples_per_subject,
    random_state=42,
    random_state_examples=0)

# === Print the shape of the datasets ===
print(f"Sample MMLU shape: {sample_mmlu.shape}")
print(f"Sample Examples MMLU shape: {sample_examples_mmlu.shape}")
print("Subjects in sample_mmlu:", sample_mmlu["subject"].nunique())
print("Subjects in sample_examples_mmlu:", sample_examples_mmlu["subject"].nunique())

#print("Number of samples per subject:\n", sample_mmlu["subject"].value_counts())
#print("Number of samples examples per subject:\n", sample_examples_mmlu["subject"].value_counts())


(5013, 5)
                                            question         subject  \
0   Just war theory's principle of military neces...  moral_disputes   
1   According to Mill, censoring speech that is p...  moral_disputes   
2             West argues that feminist rhetoric has  moral_disputes   
3   According to Mill, the value of a particular ...  moral_disputes   
4   According to Carruthers, whenever someone is ...  moral_disputes   

                                             choices  answer   category  
0  [jus in bello., jus ad bellum., moral nihilism...       0  normative  
1  [violates human dignity., fails a prima facie ...       2  normative  
2  [obscures the harms of noncoerced, consensual ...       0  normative  
3  [its quantity alone., its quality alone., both...       2  normative  
4  [the animal., the wider effects on human being...       1  normative  
Sample MMLU shape: (750, 7)
Sample Examples MMLU shape: (75, 7)
Subjects in sample_mmlu: 15
Subjects in sample_ex

In [34]:
# --- Imports ---
import re, unicodedata
import pandas as pd
from typing import List
from datasets import load_dataset
from data_loader import fetch_categories_mmlu, load_mmlu_dataset

# --- Helpers for robust duplicate detection ---
def _normalize_spaces(s: str) -> str:
    s = unicodedata.normalize("NFKC", str(s))
    s = s.replace("\r", "\n")
    s = re.sub(r"\s+", " ", s.strip())
    return s

def _extract_question_core(s: str) -> str:
    """Strip 'Question:' header, cut anything after 'Choices:', lowercase, collapse spaces."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = unicodedata.normalize("NFKC", str(s))
    s = re.sub(r"^\s*question\s*:\s*", "", s, flags=re.I)        # drop leading "Question:"
    s = re.split(r"\bchoices\s*:", s, flags=re.I)[0]              # cut off at "Choices:"
    return _normalize_spaces(s).lower()

def _build_keys(df: pd.DataFrame, subj_col="subject", q_col_preferred="question", q_col_fallback="text") -> pd.Series:
    if q_col_preferred in df.columns:
        q = df[q_col_preferred]
    elif q_col_fallback in df.columns:
        q = df[q_col_fallback]
    else:
        raise ValueError("Need a 'question' or 'text' column to build keys.")
    subj = df.get(subj_col, pd.Series([""] * len(df)))
    return subj.astype(str).str.lower().str.strip() + "||" + q.map(_extract_question_core)

def cap_normative_to_130(
    df: pd.DataFrame,
    subject_col: str = "subject",
    category_col: str = "category",
    normative_label: str = "normative",
    target: int = 130,
    seed: int = 42,
) -> pd.DataFrame:
    out = []
    for subj, g in df.groupby(subject_col, group_keys=False):
        cat_vals = g[category_col].unique()
        if len(cat_vals) != 1:
            print(f"⚠️ Subject {subj} has mixed categories {cat_vals}. Keeping as-is.")
            out.append(g)
            continue

        cat = cat_vals[0]
        n = len(g)
        if cat == normative_label:
            if n > target:
                trimmed = g.sample(n=target, random_state=seed)
                print(f"[Normative] {subj}: capped to {target} (from {n})")
                out.append(trimmed)
            else:
                print(f"[Normative] {subj}: only {n} available (<={target}), keeping all.")
                out.append(g)
        else:
            # control (or anything not 'normative'): keep all
            print(f"[Control] {subj}: keeping {n}")
            out.append(g)

    return pd.concat(out, ignore_index=True)

# ============================================================
# 1) NORMATIVE SUBJECTS
# ============================================================
normative_category = [
    "moral_scenarios",
    "professional_law",
]
dataset_3_subjects = normative_category

mmlu_full_df_norm = fetch_categories_mmlu(dataset_3_subjects)
samples_per_subject = 150
samples_examples_per_subject = 5

sample_mmlu_norm, sample_examples_mmlu_norm = load_mmlu_dataset(
    mmlu_full_df_norm,
    sample_per_subject=samples_per_subject,
    sample_examples_per_subject=samples_examples_per_subject,
    random_state=42,
    random_state_examples=0
)

# Load old zero-shot results ONCE
old_path = "results/openai_4.1_mini/zero_shot/classic/results_mmlu_zero_shot.csv"
df_old = pd.read_csv(old_path)

# Build keys for old results (prefers 'question', falls back to 'text')
old_keys = set(_build_keys(df_old))

# Filter the new normative pool against old results
norm_keys = _build_keys(sample_mmlu_norm, subj_col="subject", q_col_preferred="question")
mask_keep_norm = ~norm_keys.isin(old_keys)
sample_mmlu_normative_full = sample_mmlu_norm[mask_keep_norm].reset_index(drop=True)

print(f"[Normative] original: {len(sample_mmlu_norm)}, kept: {len(sample_mmlu_normative_full)}, removed overlaps: {(~mask_keep_norm).sum()}")

# ============================================================
# 2) CONTROL SUBJECTS
# ============================================================
control_category = [
    "college_mathematics",
    "formal_logic",
]
dataset_3_subjects = control_category

mmlu_full_df_ctrl = fetch_categories_mmlu(dataset_3_subjects)
samples_per_subject = 150
samples_examples_per_subject = 5

sample_mmlu_ctrl, sample_examples_mmlu_ctrl = load_mmlu_dataset(
    mmlu_full_df_ctrl,
    sample_per_subject=samples_per_subject,
    sample_examples_per_subject=samples_examples_per_subject,
    random_state=42,
    random_state_examples=0
)

# Filter the new control pool against old results
ctrl_keys = _build_keys(sample_mmlu_ctrl, subj_col="subject", q_col_preferred="question")
mask_keep_ctrl = ~ctrl_keys.isin(old_keys)
sample_mmlu_control_full = sample_mmlu_ctrl[mask_keep_ctrl].reset_index(drop=True)

print(f"[Control]    original: {len(sample_mmlu_ctrl)}, kept: {len(sample_mmlu_control_full)}, removed overlaps: {(~mask_keep_ctrl).sum()}")

# ============================================================
# 3) COMBINE & SAVE
# ============================================================
full_examples_df = pd.concat([sample_examples_mmlu_ctrl, sample_examples_mmlu_norm], ignore_index=True)
full_df = pd.concat([sample_mmlu_control_full, sample_mmlu_normative_full], ignore_index=True)

full_df_capped = cap_normative_to_130(full_df)
out_path = "data/data_file.csv"
full_df_capped.to_csv(out_path, index=False)
print(f"✅ Saved filtered combined dataset to: {out_path}")

# Optional: show per-subject counts after filtering
print("\nCounts after filtering:")
print(full_df_capped["subject"].value_counts())


[Normative] original: 300, kept: 270, removed overlaps: 30
[Control]    original: 216, kept: 119, removed overlaps: 97
[Control] college_mathematics: keeping 47
[Control] formal_logic: keeping 72
[Normative] moral_scenarios: capped to 130 (from 136)
[Normative] professional_law: capped to 130 (from 134)
✅ Saved filtered combined dataset to: data/data_file.csv

Counts after filtering:
subject
moral_scenarios        130
professional_law       130
formal_logic            72
college_mathematics     47
Name: count, dtype: int64


In [37]:
from dotenv import load_dotenv
import openai
import os, torch, numpy as np

load_dotenv()

ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
    "API_KEY_AZURE_OPENAI": "Azure OpenAI",
    "ENDPOINT_AZURE_OPENAI": "Azure OpenAI Endpoint",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        raise ValueError(f"Missing {name} API key: `{var}` must be set in the environment.")

client = openai.OpenAI(api_key=os.getenv("API_KEY_OPENAI"))
model = "gpt-4.1-mini"
model_filename = "openai_4.1_mini"

# Zero-shot

In [ ]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.mmlu_case import mmlu_case
from zero_shot import ZeroShot
from sklearn.metrics import classification_report, confusion_matrix
from openai import RateLimitError
from data_loader import get_additional_fields


case = mmlu_case
case_name = "mmlu"
data = sample_mmlu
output_file = f"results/{model_filename}/zero_shot/classic/results_mmlu_zero_shot.csv"

os.makedirs(os.path.dirname(output_file), exist_ok=True)

if os.path.exists(output_file):
    df_out_existing = pd.read_csv(output_file)
    done_ids = set(df_out_existing["sample_id"])
    rows = df_out_existing.to_dict(orient="records")
    print(f"=== Resuming from last index... {len(done_ids)} samples already completed.")
else:
    done_ids = set()
    rows = []

zero_shot_classifier = ZeroShot(
    case=case,
    client=client,
    model=model,
    max_tokens=300,
)

try:
    for idx, row in tqdm(data.iterrows(), total=len(data)):
        if idx in done_ids:
            continue

        text = row[case.input_col]
        true_label = row[case.label_col]
        if isinstance(true_label, str):
            true_label = true_label.strip()

        try:
            predicted_label, stats = zero_shot_classifier.classify(text)
            mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
        except RateLimitError as e:
            print(f"\nRate limit hit at sample {idx}. Saving progress.")
            break
        except Exception as e:
            print(f"\nError at sample {idx}: {e}. Skipping.")
            continue

        additional = get_additional_fields(row, case_name) 

        results = {
            "sample_id": idx,
            "text": text,
            "true_label": true_label,
            "pred_label": mapped_label,
            "max_tokens": zero_shot_classifier.max_tokens,
            "tokens_used": stats["tokens_used"],
            "prompt_tokens": stats["prompt_tokens"],
            "completion_tokens": stats["completion_tokens"],
            "latency": stats["latency"],
            **additional,
        }

        rows.append(results)

except KeyboardInterrupt:
    print("=== Interrupted manually. Saving progress...")

finally:
    df_out = pd.DataFrame(rows)
    df_out.to_csv(output_file, index=False)
    print(f"✅ Saved {len(df_out)} rows to {output_file}")
    
    y_true = df_out["true_label"].astype(int)
    y_pred = df_out["pred_label"].astype(int)

    print("=== Classification Report ===\n")
    print(classification_report(y_true, y_pred))

    print("\n=== Confusion Matrix ===\n")
    labels = sorted(set(y_true) | set(y_pred))
    conf_matrix = confusion_matrix(y_true, y_pred)
    print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

    accuracy = (y_true == y_pred).mean()
    print(f"\n=== Accuracy: {accuracy:.2%} ===")

100%|██████████| 750/750 [07:37<00:00,  1.64it/s]


✅ Saved 750 rows to results/openai_4.1_mini/zero_shot/classic/results_mmlu_zero_shot.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.77      0.78      0.77       181
           1       0.78      0.81      0.80       182
           2       0.79      0.80      0.79       188
           3       0.86      0.80      0.83       199

    accuracy                           0.80       750
   macro avg       0.80      0.80      0.80       750
weighted avg       0.80      0.80      0.80       750


=== Confusion Matrix ===

     0    1    2    3
0  141   20   12    8
1   16  148   12    6
2   14   11  151   12
3   13   10   17  159

=== Accuracy: 79.87% ===


In [ ]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.mmlu_case import mmlu_case
from zero_shot import ZeroShot
from sklearn.metrics import classification_report, confusion_matrix
from openai import RateLimitError
from data_loader import get_additional_fields


case = mmlu_case
case_name = "mmlu"
data = full_df_capped
output_file = f"results/{model_filename}/zero_shot/classic/results_mmlu_zero_shot_full.csv"

df2 = pd.read_csv("results/openai_4.1_mini/zero_shot/classic/results_mmlu_zero_shot.csv")

os.makedirs(os.path.dirname(output_file), exist_ok=True)

if os.path.exists(output_file):
    df_out_existing = pd.read_csv(output_file)
    done_ids = set(df_out_existing["sample_id"])
    rows = df_out_existing.to_dict(orient="records")
    print(f"=== Resuming from last index... {len(done_ids)} samples already completed.")
else:
    done_ids = set()
    rows = []

zero_shot_classifier = ZeroShot(
    case=case,
    client=client,
    model=model,
    max_tokens=300,
)

try:
    for idx, row in tqdm(data.iterrows(), total=len(data)):
        if idx in done_ids:
            continue

        text = row[case.input_col]
        true_label = row[case.label_col]
        if isinstance(true_label, str):
            true_label = true_label.strip()

        try:
            predicted_label, stats = zero_shot_classifier.classify(text)
            mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
        except RateLimitError as e:
            print(f"\nRate limit hit at sample {idx}. Saving progress.")
            break
        except Exception as e:
            print(f"\nError at sample {idx}: {e}. Skipping.")
            continue

        additional = get_additional_fields(row, case_name) 

        results = {
            "sample_id": idx,
            "text": text,
            "true_label": true_label,
            "pred_label": mapped_label,
            "max_tokens": zero_shot_classifier.max_tokens,
            "tokens_used": stats["tokens_used"],
            "prompt_tokens": stats["prompt_tokens"],
            "completion_tokens": stats["completion_tokens"],
            "latency": stats["latency"],
            **additional,
        }

        rows.append(results)

except KeyboardInterrupt:
    print("=== Interrupted manually. Saving progress...")

finally:
    df_out_prev = pd.DataFrame(rows)
    df_out = pd.concat([df2, df_out_prev], ignore_index=True)
    df_out.to_csv(output_file, index=False)
    print(f"✅ Saved {len(df_out)} rows to {output_file}")
    
    y_true = df_out["true_label"].astype(int)
    y_pred = df_out["pred_label"].astype(int)

    print("=== Classification Report ===\n")
    print(classification_report(y_true, y_pred))

    print("\n=== Confusion Matrix ===\n")
    labels = sorted(set(y_true) | set(y_pred))
    conf_matrix = confusion_matrix(y_true, y_pred)
    print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

    accuracy = (y_true == y_pred).mean()
    print(f"\n=== Accuracy: {accuracy:.2%} ===")

100%|██████████| 379/379 [03:18<00:00,  1.91it/s]

✅ Saved 1129 rows to results/openai_4.1_mini/zero_shot/classic/results_mmlu_zero_shot_full.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.74      0.70      0.72       284
           1       0.72      0.76      0.74       262
           2       0.75      0.75      0.75       281
           3       0.78      0.78      0.78       302

    accuracy                           0.75      1129
   macro avg       0.75      0.75      0.75      1129
weighted avg       0.75      0.75      0.75      1129


=== Confusion Matrix ===

     0    1    2    3
0  198   41   28   17
1   26  200   18   18
2   21   19  211   30
3   23   19   25  235

=== Accuracy: 74.76% ===


In [57]:
subjects_to_keep = ["moral_scenarios", "professional_law", "college_mathematics", "formal_logic"]

# Filter df2 upfront if needed
if "subject" in df2.columns and subjects_to_keep:
    df2_filtered = df2[df2["subject"].isin(subjects_to_keep)].reset_index(drop=True)

# Enforce column order
desired_order = [
    "sample_id", "text", "true_label", "pred_label",
    "max_tokens", "tokens_used", "prompt_tokens", "completion_tokens", "latency",
    "subject", "category"
]

final_cols = [c for c in desired_order if c in df_out.columns] + \
             [c for c in df_out.columns if c not in desired_order]

df_out = df_out[final_cols]

# Filter and reset index
df_out_filt = df_out[df_out["subject"].isin(subjects_to_keep)].reset_index(drop=True)

# 🔑 Rebuild sample_id from index
df_out_filt["sample_id"] = df_out_filt.index

print(df_out_filt.value_counts("subject"))

# Save once only
df_out_filt.to_csv(output_file, index=False)
print(f"✅ Saved {len(df_out_filt)} rows to {output_file}")


subject
moral_scenarios        180
professional_law       180
formal_logic           122
college_mathematics     97
Name: count, dtype: int64
✅ Saved 579 rows to results/openai_4.1_mini/zero_shot/classic/results_mmlu_zero_shot_full.csv


In [58]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.mmlu_case import mmlu_case
from zero_shot import ZeroShot
from few_shots import FewShot
from sklearn.metrics import classification_report, confusion_matrix
from openai import RateLimitError
from data_loader import get_additional_fields

case = mmlu_case
case_name = "mmlu"
data = full_df_capped

output_file_prefix = "few_shot/classic/results_mmlu_few_shot_3_shot"

output_file = f"results/{model_filename}/{output_file_prefix}_full.csv"

df2 = pd.read_csv(f"results/{model_filename}/{output_file_prefix}.csv")

subjects_to_keep = ["moral_scenarios", "professional_law", "college_mathematics", "formal_logic"]
if "subject" in df2.columns and subjects_to_keep:
    df2_filtered = df2[df2["subject"].isin(subjects_to_keep)].reset_index(drop=True)
else:
    df2_filtered = df2

os.makedirs(os.path.dirname(output_file), exist_ok=True)

if os.path.exists(output_file):
    df_out_existing = pd.read_csv(output_file)
    done_ids = set(df_out_existing["sample_id"])
    rows = df_out_existing.to_dict(orient="records")
    print(f"=== Resuming from last index... {len(done_ids)} samples already completed.")
else:
    done_ids = set()
    rows = []

if True:
    zero_shot_classifier = FewShot(
        case=case,
        client=client,
        model=model,
        max_tokens=300,
        n_shots=3,
        examples_df=full_examples_df
    )
if False:
    zero_shot_classifier = ZeroShot(
        case=case,
        client=client,
        model=model,
        max_tokens=300,
        
    )
    
try:
    for idx, row in tqdm(data.iterrows(), total=len(data)):
        if idx in done_ids:
            continue

        text = row[case.input_col]
        true_label = row[case.label_col]
        if isinstance(true_label, str):
            true_label = true_label.strip()

        try:
            predicted_label, stats = zero_shot_classifier.classify(text)
            mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
        except RateLimitError as e:
            print(f"\nRate limit hit at sample {idx}. Saving progress.")
            break
        except Exception as e:
            print(f"\nError at sample {idx}: {e}. Skipping.")
            continue

        additional = get_additional_fields(row, case_name)

        results = {
            "sample_id": idx,
            "text": text,
            "true_label": true_label,
            "pred_label": mapped_label,
            "max_tokens": zero_shot_classifier.max_tokens,
            "tokens_used": stats["tokens_used"],
            "prompt_tokens": stats["prompt_tokens"],
            "completion_tokens": stats["completion_tokens"],
            "latency": stats["latency"],
            **additional,
        }

        rows.append(results)

except KeyboardInterrupt:
    print("=== Interrupted manually. Saving progress...")

finally:
    df_out_prev = pd.DataFrame(rows)
    df_out = pd.concat([df2_filtered, df_out_prev], ignore_index=True)

    desired_order = [
        "sample_id", "text", "true_label", "pred_label",
        "max_tokens", "tokens_used", "prompt_tokens", "completion_tokens", "latency",
        "subject", "category"
    ]

    final_cols = [c for c in desired_order if c in df_out.columns] + \
                 [c for c in df_out.columns if c not in desired_order]

    df_out = df_out[final_cols]

    df_out_filt = df_out[df_out["subject"].isin(subjects_to_keep)].reset_index(drop=True)
    df_out_filt["sample_id"] = df_out_filt.index

    print(df_out_filt.value_counts("subject"))

    df_out_filt.to_csv(output_file, index=False)
    print(f"✅ Saved {len(df_out_filt)} rows to {output_file}")

    y_true = df_out_filt["true_label"].astype(int)
    y_pred = df_out_filt["pred_label"].astype(int)

    print("=== Classification Report ===\n")
    print(classification_report(y_true, y_pred))

    print("\n=== Confusion Matrix ===\n")
    labels = sorted(set(y_true) | set(y_pred))
    conf_matrix = confusion_matrix(y_true, y_pred)
    print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

    accuracy = (y_true == y_pred).mean()
    print(f"\n=== Accuracy: {accuracy:.2%} ===")


100%|██████████| 379/379 [03:41<00:00,  1.71it/s]

subject
moral_scenarios        180
professional_law       180
formal_logic           122
college_mathematics     97
Name: count, dtype: int64
✅ Saved 579 rows to results/openai_4.1_mini/few_shot/classic/results_mmlu_few_shot_3_shot_full.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.67      0.69      0.68       151
           1       0.62      0.65      0.64       124
           2       0.66      0.54      0.59       138
           3       0.66      0.72      0.69       166

    accuracy                           0.65       579
   macro avg       0.65      0.65      0.65       579
weighted avg       0.65      0.65      0.65       579


=== Confusion Matrix ===

     0   1   2    3
0  104  23  11   13
1   17  81  11   15
2   19  12  74   33
3   16  14  16  120

=== Accuracy: 65.46% ===


# Few-shots

In [ ]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.mmlu_case import mmlu_case
from few_shots import FewShot
from sklearn.metrics import classification_report, confusion_matrix
from data_loader import get_additional_fields


case = mmlu_case
case_name = "mmlu"
data = sample_mmlu
examples_df = sample_examples_mmlu


for j in range(2, 6):
    output_file = f"results/{model_filename}/few_shot/classic/results_mmlu_few_shot_{j}_shot.csv"
    
    
    few_shot_classifier = FewShot(
        case=case,
        client=client,
        model=model,
        max_tokens=300,
        task_definition=None, 
        n_shots=j,
        examples_df=examples_df,
    )
    
    
    rows = []
    
    
    for idx, row in tqdm(data.iterrows(), total=len(data)):
        text = row[case.input_col]
        true_label = row[case.label_col]
        
        if isinstance(true_label, str):
            true_label = true_label.strip()
    
        predicted_label, stats = few_shot_classifier.classify(text, row=row)
        mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
    
        additional = get_additional_fields(row, case_name)
    
        results = {
            "sample_id": idx,
            "text": text,
            "true_label": true_label,
            "pred_label": mapped_label,
            "max_tokens": few_shot_classifier.max_tokens,
            "tokens_used": stats["tokens_used"],
            "prompt_tokens": stats["prompt_tokens"],
            "completion_tokens": stats["completion_tokens"],
            "latency": stats["latency"],
            **additional,
        }
    
        rows.append(results)
    
    
    df_out = pd.DataFrame(rows)
    df_out.to_csv(output_file, index=False)
    print(f"=== Saved {len(df_out)} rows to {output_file}")
    
    y_true = df_out["true_label"].astype(int)
    y_pred = df_out["pred_label"].astype(int)

    print("=== Classification Report ===\n")
    print(classification_report(y_true, y_pred))
    
    
    print("\n=== Confusion Matrix ===\n")
    labels = sorted(set(y_true) | set(y_pred))
    conf_matrix = confusion_matrix(y_true, y_pred)
    print(pd.DataFrame(conf_matrix, index=labels, columns=labels))
    
    accuracy = (y_true == y_pred).mean()
    print(f"\n=== Accuracy: {accuracy:.2%} ===")

## Role-playing in Zero-shot and Few-shots settings (3 examples)

In [8]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.mmlu_case import mmlu_case
from zero_shot import ZeroShot
from few_shots import FewShot
from sklearn.metrics import classification_report, confusion_matrix
from openai import RateLimitError
from data_loader import get_additional_fields
from profiles.profile_sets import PERSON_ETHNICS


selected_profiles = [f"profile{i}" for i in range(41, 61)]

case = mmlu_case
case_name = "mmlu"
data = sample_mmlu
max_tokens = 300

role_playing = "passive"
person_set = PERSON_ETHNICS


for person_key in selected_profiles:

    output_file = f"results/{model_filename}/few_shot/role_playing_ethnics/{person_key}_{role_playing}/results_mmlu_few_shot_3examples.csv"

    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    if os.path.exists(output_file):
        df_out_existing = pd.read_csv(output_file)
        done_ids = set(df_out_existing["sample_id"])
        rows = df_out_existing.to_dict(orient="records")
        print(f"=== Resuming from last index... {len(done_ids)} samples already completed.")
    else:
        done_ids = set()
        rows = []
    
    zero_shot_classifier = FewShot(
        case=case,
        client=client,
        model=model,
        max_tokens=max_tokens,
        person_key=person_key,
        role_playing=role_playing,
        person_set=person_set,
        examples_df=sample_examples_mmlu
    )
    
    try:
        for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"mmlu | {person_key} | few-shot | {role_playing}"):
            if idx in done_ids:
                continue
    
            text = row[case.input_col]
            true_label = row[case.label_col]
            if isinstance(true_label, str):
                true_label = true_label.strip()
    
            try:
                predicted_label, stats = zero_shot_classifier.classify(text)
                mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
            except RateLimitError as e:
                print(f"Error : {e}")
                print(f"WARNING: Rate limit hit at sample {idx}. Skipping.")
                continue
            except Exception as e:
                print(f"ERROR at sample {idx}: {e}")
                continue
    
            additional = get_additional_fields(row, case_name) 
    
            results = {
                "sample_id": idx,
                "text": text,
                "true_label": true_label,
                "pred_label": mapped_label,
                "max_tokens": zero_shot_classifier.max_tokens,
                "tokens_used": stats["tokens_used"],
                "prompt_tokens": stats["prompt_tokens"],
                "completion_tokens": stats["completion_tokens"],
                "latency": stats["latency"],
                **additional,
            }
    
            rows.append(results)
    
    except KeyboardInterrupt:
        print("=== Interrupted manually. Saving progress...")
    
    finally:
        df_out = pd.DataFrame(rows)
        df_out.to_csv(output_file, index=False)
        print(f"✅ Saved {len(df_out)} rows to {output_file}")
        
        y_true = df_out["true_label"].astype(int)
        y_pred = df_out["pred_label"].astype(int)
    
        print("=== Classification Report ===\n")
        print(classification_report(y_true, y_pred))
    
        print("\n=== Confusion Matrix ===\n")
        labels = sorted(set(y_true) | set(y_pred))
        conf_matrix = confusion_matrix(y_true, y_pred)
        print(pd.DataFrame(conf_matrix, index=labels, columns=labels))
    
        accuracy = (y_true == y_pred).mean()
        print(f"\n=== Accuracy for mmlu | {person_key} | {role_playing}: {accuracy:.2%} ===")

=== Resuming from last index... 601 samples already completed.


mmlu | profile41 | few-shot | passive: 100%|██████████| 750/750 [01:24<00:00,  8.89it/s] 


✅ Saved 750 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile41_passive/results_mmlu_few_shot_3examples.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.76      0.75      0.76       181
           1       0.75      0.81      0.78       182
           2       0.79      0.78      0.78       188
           3       0.82      0.78      0.80       199

    accuracy                           0.78       750
   macro avg       0.78      0.78      0.78       750
weighted avg       0.78      0.78      0.78       750


=== Confusion Matrix ===

     0    1    2    3
0  136   22   13   10
1   17  147    9    9
2   13   14  146   15
3   13   14   17  155

=== Accuracy for mmlu | profile41 | passive: 77.87% ===


mmlu | profile42 | few-shot | passive: 100%|██████████| 750/750 [07:10<00:00,  1.74it/s]


✅ Saved 750 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile42_passive/results_mmlu_few_shot_3examples.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.76      0.75      0.76       181
           1       0.76      0.80      0.78       182
           2       0.79      0.78      0.79       188
           3       0.82      0.79      0.81       199

    accuracy                           0.78       750
   macro avg       0.78      0.78      0.78       750
weighted avg       0.78      0.78      0.78       750


=== Confusion Matrix ===

     0    1    2    3
0  136   22   14    9
1   18  146    8   10
2   12   13  147   16
3   13   11   17  158

=== Accuracy for mmlu | profile42 | passive: 78.27% ===


...

✅ Saved 750 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile58_passive/results_mmlu_few_shot_3examples.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.76      0.77      0.76       181
           1       0.76      0.81      0.78       182
           2       0.79      0.76      0.77       188
           3       0.83      0.79      0.81       199

    accuracy                           0.78       750
   macro avg       0.78      0.78      0.78       750
weighted avg       0.78      0.78      0.78       750


=== Confusion Matrix ===

     0    1    2    3
0  139   21   11   10
1   16  148   11    7
2   16   14  142   16
3   12   13   16  158

=== Accuracy for mmlu | profile58 | passive: 78.27% ===


mmlu | profile59 | few-shot | passive: 100%|██████████| 750/750 [06:19<00:00,  1.98it/s]


✅ Saved 750 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile59_passive/results_mmlu_few_shot_3examples.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.75      0.76      0.76       181
           1       0.76      0.81      0.78       182
           2       0.80      0.77      0.78       188
           3       0.83      0.81      0.82       199

    accuracy                           0.79       750
   macro avg       0.79      0.79      0.79       750
weighted avg       0.79      0.79      0.79       750


=== Confusion Matrix ===

     0    1    2    3
0  138   21   12   10
1   17  147    9    9
2   15   14  144   15
3   13   11   14  161

=== Accuracy for mmlu | profile59 | passive: 78.67% ===


mmlu | profile60 | few-shot | passive: 100%|██████████| 750/750 [06:15<00:00,  2.00it/s]

✅ Saved 750 rows to results/openai_4.1_mini/few_shot/role_playing_ethnics/profile60_passive/results_mmlu_few_shot_3examples.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.76      0.76      0.76       181
           1       0.75      0.80      0.77       182
           2       0.78      0.76      0.77       188
           3       0.82      0.80      0.81       199

    accuracy                           0.78       750
   macro avg       0.78      0.78      0.78       750
weighted avg       0.78      0.78      0.78       750


=== Confusion Matrix ===

     0    1    2    3
0  137   20   14   10
1   17  145   11    9
2   14   15  143   16
3   12   13   15  159

=== Accuracy for mmlu | profile60 | passive: 77.87% ===


In [41]:
df_out = pd.read_csv("results/openai_4.1_mini/few_shot/classic/results_mmlu_few_shot_4_shot.csv")

y_true = df_out["true_label"].astype(int)
y_pred = df_out["pred_label"].astype(int)
print("=== Classification Report ===\n")
print(classification_report(y_true, y_pred))
print("\n=== Confusion Matrix ===\n")
labels = sorted(set(y_true) | set(y_pred))
conf_matrix = confusion_matrix(y_true, y_pred)
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

accuracy = (y_true == y_pred).mean()
print(f"\n=== Accuracy: {accuracy:.2%} ===")

df_out_norm = df_out[df_out["category"]=="normative"]

y_true_norm = df_out_norm["true_label"].astype(int)
y_pred_norm = df_out_norm["pred_label"].astype(int)
print("=== Classification Report ===\n")
print(classification_report(y_true_norm, y_pred_norm))
print("\n=== Confusion Matrix ===\n")
labels = sorted(set(y_true_norm) | set(y_pred_norm))
conf_matrix = confusion_matrix(y_true_norm, y_pred_norm)
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

accuracy = (y_true_norm == y_pred_norm).mean()
print(f"\n=== Accuracy: {accuracy:.2%} ===")

df_out_control = df_out[df_out["category"] == "control"]

y_true_control = df_out_control["true_label"].astype(int)
y_pred_control = df_out_control["pred_label"].astype(int)

print("=== Classification Report ===\n")
print(classification_report(y_true_control, y_pred_control))

print("\n=== Confusion Matrix ===\n")
labels = sorted(set(y_true_control) | set(y_pred_control))
conf_matrix = confusion_matrix(y_true_control, y_pred_control)
print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

accuracy = (y_true_control == y_pred_control).mean()
print(f"\n=== Accuracy: {accuracy:.2%} ===")

=== Classification Report ===

              precision    recall  f1-score   support

           0       0.77      0.81      0.79       181
           1       0.78      0.82      0.80       182
           2       0.82      0.79      0.81       188
           3       0.84      0.79      0.82       199

    accuracy                           0.80       750
   macro avg       0.80      0.80      0.80       750
weighted avg       0.80      0.80      0.80       750


=== Confusion Matrix ===

     0    1    2    3
0  147   18    8    8
1   15  149    8   10
2   16   12  149   11
3   13   12   17  157

=== Accuracy: 80.27% ===
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.82      0.88      0.85       123
           1       0.84      0.90      0.87       127
           2       0.91      0.85      0.88       132
           3       0.90      0.84      0.87       118

    accuracy                           0.87       500
   macro avg  

# Roleplaying

In [ ]:
import pandas as pd
from tqdm import tqdm
import os, json
from cases.mmlu_case import mmlu_case
from zero_shot import ZeroShot
from few_shots import FewShot
from sklearn.metrics import classification_report, confusion_matrix
from openai import RateLimitError
from data_loader import get_additional_fields
from profiles.profile_sets import PERSON_ETHNICS

case = mmlu_case
case_name = "mmlu"
data = full_df_capped
output_file_prefix = "few_shot/classic/results_mmlu_few_shot_3_shot"
df2 = pd.read_csv(f"results/{model_filename}/{output_file_prefix}.csv")

subjects_to_keep = ["moral_scenarios", "professional_law", "college_mathematics", "formal_logic"]
if "subject" in df2.columns and subjects_to_keep:
    df2_filtered = df2[df2["subject"].isin(subjects_to_keep)].reset_index(drop=True)
else:
    df2_filtered = df2


a = [f"profile{i}" for i in range(41, 61)]
b = [f"profile{i}" for i in range(1, 5)]
c = [f"profile{i}" for i in range(6, 10)]
d = [f"profile{i}" for i in range(21, 25)]
e = [f"profile{i}" for i in range(26, 30)]
selected_profiles = a+b+c+d+e
role_playing = "passive"
person_set = PERSON_ETHNICS

USE_FEW_SHOT = False
USE_ZERO_SHOT = True

for person_key in selected_profiles:
    if USE_FEW_SHOT:
        output_file = f"results/{model_filename}/few_shot/role_playing_ethnics/{person_key}_{role_playing}/results_mmlu_few_shot_3examples_full.csv"
    else:
        output_file = f"results/{model_filename}/zero_shot/role_playing_ethnics/{person_key}_{role_playing}/results_mmlu_zero_shot_full.csv"

    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    if os.path.exists(output_file):
        df_out_existing = pd.read_csv(output_file)
        done_ids = set(df_out_existing["sample_id"])
        rows = df_out_existing.to_dict(orient="records")
        print(f"=== Resuming from last index... {len(done_ids)} samples already completed.")
    else:
        done_ids = set()
        rows = []

    if USE_FEW_SHOT:
        zero_shot_classifier = FewShot(
            case=case,
            client=client,
            model=model,
            max_tokens=300,
            n_shots=3,
            examples_df=full_examples_df,
            person_key=person_key,
            role_playing=role_playing,
            person_set=person_set
        )
        loop_desc = f"mmlu | {person_key} | few-shot | {role_playing}"
    elif USE_ZERO_SHOT:
        zero_shot_classifier = ZeroShot(
            case=case,
            client=client,
            model=model,
            max_tokens=300,
            person_key=person_key,
            role_playing=role_playing,
            person_set=person_set
        )
        loop_desc = f"mmlu | {person_key} | zero-shot | {role_playing}"
    else:
        raise ValueError("Set USE_FEW_SHOT or USE_ZERO_SHOT to True.")

    try:
        for idx, row in tqdm(data.iterrows(), total=len(data), desc=loop_desc):
            if idx in done_ids:
                continue

            text = row[case.input_col]
            true_label = row[case.label_col]
            if isinstance(true_label, str):
                true_label = true_label.strip()

            try:
                predicted_label, stats = zero_shot_classifier.classify(text)
                mapped_label = case.label_map.get(predicted_label.strip(), list(case.label_map.values())[-1])
            except RateLimitError as e:
                print(f"Error : {e}")
                print(f"WARNING: Rate limit hit at sample {idx}. Skipping.")
                continue
            except Exception as e:
                print(f"ERROR at sample {idx}: {e}")
                continue

            additional = get_additional_fields(row, case_name)

            results = {
                "sample_id": idx,
                "text": text,
                "true_label": true_label,
                "pred_label": mapped_label,
                "max_tokens": zero_shot_classifier.max_tokens,
                "tokens_used": stats["tokens_used"],
                "prompt_tokens": stats["prompt_tokens"],
                "completion_tokens": stats["completion_tokens"],
                "latency": stats["latency"],
                **additional,
            }

            rows.append(results)

    except KeyboardInterrupt:
        print("=== Interrupted manually. Saving progress...")

    finally:
        df_out_prev = pd.DataFrame(rows)
        df_out = pd.concat([df2_filtered, df_out_prev], ignore_index=True)

        desired_order = [
            "sample_id", "text", "true_label", "pred_label",
            "max_tokens", "tokens_used", "prompt_tokens", "completion_tokens", "latency",
            "subject", "category"
        ]

        final_cols = [c for c in desired_order if c in df_out.columns] + \
                     [c for c in df_out.columns if c not in desired_order]

        df_out = df_out[final_cols]

        df_out_filt = df_out[df_out["subject"].isin(subjects_to_keep)].reset_index(drop=True)
        df_out_filt["sample_id"] = df_out_filt.index

        print(df_out_filt.value_counts("subject"))

        df_out_filt.to_csv(output_file, index=False)
        print(f"✅ Saved {len(df_out_filt)} rows to {output_file}")

        y_true = df_out_filt["true_label"].astype(int)
        y_pred = df_out_filt["pred_label"].astype(int)

        print("=== Classification Report ===\n")
        print(classification_report(y_true, y_pred))

        print("\n=== Confusion Matrix ===\n")
        labels = sorted(set(y_true) | set(y_pred))
        conf_matrix = confusion_matrix(y_true, y_pred)
        print(pd.DataFrame(conf_matrix, index=labels, columns=labels))

        accuracy = (y_true == y_pred).mean()
        print(f"\n=== Accuracy for {loop_desc}: {accuracy:.2%} ===")


=== Resuming from last index... 750 samples already completed.


mmlu | profile41 | zero-shot | passive: 100%|██████████| 379/379 [00:00<00:00, 33462.61it/s]


subject
college_mathematics    100
formal_logic           100
moral_scenarios        100
professional_law       100
Name: count, dtype: int64
✅ Saved 400 rows to results/openai_4.1_mini/zero_shot/role_playing_ethnics/profile41_passive/results_mmlu_zero_shot.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.60      0.55      0.58        96
           1       0.56      0.66      0.60        88
           2       0.60      0.64      0.62        90
           3       0.75      0.66      0.70       126

    accuracy                           0.63       400
   macro avg       0.63      0.63      0.63       400
weighted avg       0.64      0.63      0.63       400


=== Confusion Matrix ===

    0   1   2   3
0  53  26  10   7
1  12  58  10   8
2  11   8  58  13
3  12  12  19  83

=== Accuracy for mmlu | profile41 | zero-shot | passive: 63.00% ===
=== Resuming from last index... 750 samples already completed.


mmlu | profile42 | zero-shot | passive: 100%|██████████| 379/379 [00:00<00:00, 37532.26it/s]


subject
college_mathematics    100
formal_logic           100
moral_scenarios        100
professional_law       100
Name: count, dtype: int64
✅ Saved 400 rows to results/openai_4.1_mini/zero_shot/role_playing_ethnics/profile42_passive/results_mmlu_zero_shot.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.59      0.52      0.55        96
           1       0.58      0.69      0.63        88
           2       0.60      0.67      0.63        90
           3       0.73      0.63      0.68       126

    accuracy                           0.63       400
   macro avg       0.62      0.63      0.62       400
weighted avg       0.63      0.63      0.63       400


=== Confusion Matrix ===

    0   1   2   3
0  50  27  11   8
1  10  61   8   9
2  11   7  60  12
3  14  11  21  80

=== Accuracy for mmlu | profile42 | zero-shot | passive: 62.75% ===
=== Resuming from last index... 750 samples already completed.


...

subject
college_mathematics    100
formal_logic           100
moral_scenarios        100
professional_law       100
Name: count, dtype: int64
✅ Saved 400 rows to results/openai_4.1_mini/zero_shot/role_playing_ethnics/profile27_passive/results_mmlu_zero_shot.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.58      0.52      0.55        96
           1       0.56      0.66      0.60        88
           2       0.62      0.67      0.65        90
           3       0.74      0.67      0.70       126

    accuracy                           0.63       400
   macro avg       0.63      0.63      0.62       400
weighted avg       0.63      0.63      0.63       400


=== Confusion Matrix ===

    0   1   2   3
0  50  28  12   6
1  12  58   7  11
2  10   7  60  13
3  14  11  17  84

=== Accuracy for mmlu | profile27 | zero-shot | passive: 63.00% ===
=== Resuming from last index... 750 samples already completed.


mmlu | profile28 | zero-shot | passive: 100%|██████████| 379/379 [00:00<00:00, 35526.68it/s]


subject
college_mathematics    100
formal_logic           100
moral_scenarios        100
professional_law       100
Name: count, dtype: int64
✅ Saved 400 rows to results/openai_4.1_mini/zero_shot/role_playing_ethnics/profile28_passive/results_mmlu_zero_shot.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.57      0.51      0.54        96
           1       0.56      0.66      0.60        88
           2       0.59      0.67      0.62        90
           3       0.74      0.63      0.68       126

    accuracy                           0.62       400
   macro avg       0.61      0.62      0.61       400
weighted avg       0.63      0.62      0.62       400


=== Confusion Matrix ===

    0   1   2   3
0  49  28  11   8
1  12  58  10   8
2  11   7  60  12
3  14  11  21  80

=== Accuracy for mmlu | profile28 | zero-shot | passive: 61.75% ===
=== Resuming from last index... 750 samples already completed.


mmlu | profile29 | zero-shot | passive: 100%|██████████| 379/379 [00:00<00:00, 37398.92it/s]

subject
college_mathematics    100
formal_logic           100
moral_scenarios        100
professional_law       100
Name: count, dtype: int64
✅ Saved 400 rows to results/openai_4.1_mini/zero_shot/role_playing_ethnics/profile29_passive/results_mmlu_zero_shot.csv
=== Classification Report ===

              precision    recall  f1-score   support

           0       0.58      0.50      0.54        96
           1       0.57      0.67      0.62        88
           2       0.61      0.67      0.63        90
           3       0.73      0.67      0.70       126

    accuracy                           0.63       400
   macro avg       0.62      0.63      0.62       400
weighted avg       0.63      0.63      0.63       400


=== Confusion Matrix ===

    0   1   2   3
0  48  26  13   9
1  11  59   8  10
2  11   7  60  12
3  13  11  18  84

=== Accuracy for mmlu | profile29 | zero-shot | passive: 62.75% ===


# Test

In [ ]:
import os
import json
import pandas as pd
from typing import List, Tuple, Optional
from datasets import load_dataset

def _dedupe_key_from_row(row):
    ch = row["choices"]
    if isinstance(ch, str):
        try:
            ch = json.loads(ch)
        except Exception:
            ch = [str(ch)]
    if not isinstance(ch, list):
        ch = list(ch) if ch is not None else []
    return f"{row['subject']}||{row['question']}||{'|'.join(map(str, ch))}"

def _build_dedupe_keys(df: pd.DataFrame) -> pd.Series:
    if "choices" not in df.columns or "question" not in df.columns or "subject" not in df.columns:
        raise ValueError("DataFrame must contain columns: 'subject', 'question', 'choices'.")
    return df.apply(_dedupe_key_from_row, axis=1)

def fetch_categories_mmlu(subject_list: List[str],
                          normative_category: List[str],
                          control_category: List[str]) -> pd.DataFrame:
    dfs = []
    for sub in subject_list:
        ds = load_dataset("cais/mmlu", sub)
        if "test" not in ds:
            continue
        df = ds["test"].to_pandas()
        df["subject"] = sub
        if sub in normative_category:
            df["category"] = "normative"
        elif sub in control_category:
            df["category"] = "control"
        else:
            raise ValueError(f"Subject '{sub}' is not in normative or control lists.")
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

def prepare_deep_subjects_csvs(
    mmlu_full_df: pd.DataFrame,
    deep_normative: List[str],
    deep_control: List[str],
    existing_results_csv: str,
    out_dir: str = ".",
    n_new_per_subject: int = 150,
    seed: int = 123,
    new_items_filename: str = "deep_subjects_new_items.csv",
    existing_final_filename: str = "deep_subjects_existing_final.csv",
) -> Tuple[str, str]:
    deep_subjects = list(deep_normative) + list(deep_control)
    df_deep = mmlu_full_df[mmlu_full_df["subject"].isin(deep_subjects)].copy()
    if df_deep.empty:
        raise ValueError("No rows found for the requested deep subjects.")
    df_deep["dedupe_key"] = _build_dedupe_keys(df_deep)

    if not os.path.exists(existing_results_csv):
        raise FileNotFoundError(f"existing_results_csv not found: {existing_results_csv}")
    existing = pd.read_csv(existing_results_csv)

    if {"subject", "question"}.issubset(existing.columns) and "choices" in existing.columns:
        try:
            existing["dedupe_key"] = _build_dedupe_keys(existing)
        except Exception:
            existing["dedupe_key"] = pd.Series([None] * len(existing))
    elif "formatted_prompt" in existing.columns:
        existing["dedupe_key"] = existing["formatted_prompt"].astype(str)
        df_deep["dedupe_key"] = (
            df_deep["subject"].astype(str)
            + "||"
            + df_deep["question"].astype(str)
            + "||"
            + df_deep["choices"].apply(
                lambda x: "|".join(map(str, x)) if isinstance(x, list) else str(x)
            )
        )
    else:
        existing["dedupe_key"] = pd.Series([None] * len(existing))

    existing_deep = existing[existing["subject"].isin(deep_subjects)].copy()
    used = set(existing_deep["dedupe_key"].dropna().astype(str).tolist())

    picks = []
    for sub in deep_subjects:
        pool = df_deep[df_deep["subject"] == sub].copy()
        pool = pool[~pool["dedupe_key"].isin(used)].copy()
        available = len(pool)

        if available == 0:
            print(f"⚠️ No new examples left for subject '{sub}'. Skipping.")
            continue

        n_to_sample = min(n_new_per_subject, available)
        print(f"Subject '{sub}': picking {n_to_sample} (out of {available} available).")

        picks.append(pool.sample(n=n_to_sample, random_state=seed))

    if not picks:
        raise ValueError("No new items could be sampled for any deep subject.")

    new_items = pd.concat(picks, ignore_index=True)

    for df in (new_items,):
        df["choices"] = df["choices"].apply(
            lambda x: list(x) if isinstance(x, (list, tuple)) else (json.loads(x) if isinstance(x, str) else [])
        )
        df["answer"] = df["answer"].astype(str)

    new_items_path = os.path.join(out_dir, new_items_filename)
    existing_final_path = os.path.join(out_dir, existing_final_filename)

    new_items.to_csv(new_items_path, index=False)
    existing_deep.to_csv(existing_final_path, index=False)

    return new_items_path, existing_final_path



normative_category = [
    "moral_disputes","philosophy","world_religions","us_foreign_policy","sociology",
    "professional_psychology","professional_law","moral_scenarios","human_sexuality","international_law",
]
control_category = ["college_mathematics","college_physics","formal_logic","logical_fallacies","college_computer_science"]

deep_normative = ["professional_law","moral_scenarios"]
deep_control = ["college_mathematics","formal_logic"]

dataset_subjects = normative_category + control_category
mmlu_full_df = fetch_categories_mmlu(dataset_subjects, normative_category, control_category)

new_items_path, existing_final_path = prepare_deep_subjects_csvs(
    mmlu_full_df=mmlu_full_df,
    deep_normative=["professional_law","moral_scenarios"],
    deep_control=["college_mathematics","formal_logic"],
    existing_results_csv="results/openai_4.1_mini/zero_shot/classic/results_mmlu_zero_shot.csv",
    out_dir="data/",
    n_new_per_subject=150,
)

print("New items to run:", new_items_path)
print("Existing final (current results, 4 subjects):", existing_final_path)